# Validation & Library Comparisons

## Overview

Validation is a critical step in ensuring the accuracy and reliability of FEA simulations. This notebook covers comprehensive validation techniques, comparison with analytical solutions, and benchmarking against established FEA libraries including SfePy and FEniCSx. We'll explore various methods to verify our implementations and assess their suitability for different electromagnetic problems.

## Learning Objectives

After completing this notebook, you will be able to:
- Validate FEA results against analytical solutions
- Perform convergence studies and error analysis
- Compare different FEA libraries and solvers
- Choose appropriate libraries for specific problem types
- Implement robust validation workflows

## Topics Covered

1. **Analytical Validation** with known solutions
2. **Convergence Studies** and error analysis
3. **SfePy Implementation** for electromagnetic problems
4. **FEniCSx Comparison** and advanced features
5. **Library Performance** and suitability assessment

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay
from scipy.sparse import lil_matrix, csr_matrix
from scipy.sparse.linalg import spsolve
from scipy.interpolate import griddata
import matplotlib.tri as tri
from matplotlib.patches import Circle, Rectangle
import time
import warnings
warnings.filterwarnings('ignore')

# Set up matplotlib for better plots
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
np.set_printoptions(precision=4, suppress=True)

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"SciPy version: {np.__version__}")

# Try to import additional libraries (may not be installed)
try:
    import sfepy
    sfepy_available = True
    print(f"SfePy version: {sfepy.__version__}")
except ImportError:
    sfepy_available = False
    print("SfePy not available - will provide examples")

try:
    import dolfinx as fem
    fenics_available = True
    print(f"FEniCSx/DOLFINx available")
except ImportError:
    fenics_available = False
    print("FEniCSx not available - will provide examples")

try:
    import pygmsh
    pygmsh_available = True
    print("PyGmsh available")
except ImportError:
    pygmsh_available = False
    print("PyGmsh not available")

## 1. Analytical Validation

We'll validate our FEA implementation against problems with known analytical solutions. This is essential for verifying the correctness of our numerical methods.

In [ ]:
# ---------- 1. Analytical Validation Framework ----------
class AnalyticalSolutions:
    """
    Collection of analytical solutions for electromagnetic problems.
    """
    
    @staticmethod
    def infinite_wire_field(r, I, mu0=4*np.pi*1e-7):
        """
        Magnetic field of infinite straight wire carrying current I.
        
        B = μ₀I/(2πr)
        
        Parameters:
        -----------
        r : float or array
            Distance from wire center
        I : float
            Current [A]
        mu0 : float
            Permeability of free space
        
        Returns:
        --------
        B : float or array
            Magnetic field magnitude [T]
        """
        return mu0 * I / (2 * np.pi * r)
    
    @staticmethod
    def infinite_wire_potential(r, I, mu0=4*np.pi*1e-7):
        """
        Magnetic vector potential of infinite wire.
        
        A_z = (μ₀I/2π) * ln(r/r_ref)
        
        Parameters:
        -----------
        r : float or array
            Distance from wire center
        I : float
            Current [A]
        r_ref : float
            Reference radius (default: 1)
        mu0 : float
            Permeability of free space
        
        Returns:
        --------
        A_z : float or array
            Magnetic vector potential [Wb/m]
        """
        r_ref = 1.0  # Reference radius
        return (mu0 * I / (2 * np.pi)) * np.log(r / r_ref)
    
    @staticmethod
    def parallel_wires_field(x, y, I1, I2, d, mu0=4*np.pi*1e-7):
        """
        Magnetic field of two parallel wires.
        
        Wire 1 at (-d/2, 0) carrying current I1
        Wire 2 at (+d/2, 0) carrying current I2
        
        Parameters:
        -----------
        x, y : arrays
            Coordinate arrays
        I1, I2 : float
            Currents in wires [A]
        d : float
            Separation between wires [m]
        mu0 : float
            Permeability of free space
        
        Returns:
        --------
        Bx, By : arrays
            Magnetic field components [T]
        """
        # Distance from each wire
        r1 = np.sqrt((x + d/2)**2 + y**2)
        r2 = np.sqrt((x - d/2)**2 + y**2)
        
        # Avoid singularities
        r1 = np.maximum(r1, 1e-10)
        r2 = np.maximum(r2, 1e-10)
        
        # Field from each wire (using Biot-Savart law)
        B1_mag = mu0 * I1 / (2 * np.pi * r1)
        B2_mag = mu0 * I2 / (2 * np.pi * r2)
        
        # Components (perpendicular to r vector)
        B1x = -B1_mag * y / r1
        B1y = B1_mag * (x + d/2) / r1
        
        B2x = -B2_mag * y / r2
        B2y = B2_mag * (x - d/2) / r2
        
        return B1x + B2x, B1y + B2y
    
    @staticmethod
    def coaxial_cable_field(r, r_inner, r_outer, I, mu0=4*np.pi*1e-7):
        """
        Magnetic field in coaxial cable.
        
        Inner conductor radius: r_inner, current I
        Outer conductor radius: r_outer, current -I
        
        Parameters:
        -----------
        r : array
            Radial positions
        r_inner, r_outer : float
            Inner and outer conductor radii
        I : float
            Current in inner conductor [A]
        mu0 : float
            Permeability of free space
        
        Returns:
        --------
        B : array
            Magnetic field magnitude [T]
        """
        B = np.zeros_like(r)
        
        # Inside inner conductor (r < r_inner)
        mask1 = r < r_inner
        B[mask1] = mu0 * I * r[mask1] / (2 * np.pi * r_inner**2)
        
        # Between conductors (r_inner <= r < r_outer)
        mask2 = (r >= r_inner) & (r < r_outer)
        B[mask2] = mu0 * I / (2 * np.pi * r[mask2])
        
        # Outside outer conductor (r >= r_outer)
        # Net current is zero, so field is zero
        # B[mask3] = 0  (already zero)
        
        return B

def create_validation_mesh(domain_type='circular', n_points=500, **kwargs):
    """
    Create mesh for validation problems.
    
    Parameters:
    -----------
    domain_type : str
        Type of domain ('circular', 'rectangular', 'coaxial')
    n_points : int
        Number of points
    **kwargs : dict
        Additional parameters for specific domains
    
    Returns:
    --------
    points, elements : arrays
        Mesh coordinates and connectivity
    """
    np.random.seed(42)
    
    if domain_type == 'circular':
        radius = kwargs.get('radius', 1.0)
        
        # Generate points in circle
        points = []
        for i in range(n_points):
            r = radius * np.sqrt(np.random.rand())
            theta = np.random.rand() * 2 * np.pi
            points.append([r * np.cos(theta), r * np.sin(theta)])
        
        # Add boundary points
        n_boundary = 40
        for i in range(n_boundary):
            theta = 2 * np.pi * i / n_boundary
            points.append([radius * np.cos(theta), radius * np.sin(theta)])
        
        points = np.array(points)
        
    elif domain_type == 'rectangular':
        width = kwargs.get('width', 2.0)
        height = kwargs.get('height', 1.0)
        nx = int(np.sqrt(n_points * width/height))
        ny = int(n_points / nx)
        
        x = np.linspace(0, width, nx)
        y = np.linspace(0, height, ny)
        X, Y = np.meshgrid(x, y)
        points = np.column_stack([X.ravel(), Y.ravel()])
        
    elif domain_type == 'coaxial':
        r_inner = kwargs.get('r_inner', 0.2)
        r_outer = kwargs.get('r_outer', 0.5)
        r_domain = kwargs.get('r_domain', 1.0)
        
        points = []
        
        # Inner conductor region
        n_inner = int(n_points * 0.2)
        for i in range(n_inner):
            r = r_inner * np.sqrt(np.random.rand())
            theta = np.random.rand() * 2 * np.pi
            points.append([r * np.cos(theta), r * np.sin(theta)])
        
        # Dielectric region
        n_dielectric = int(n_points * 0.4)
        for i in range(n_dielectric):
            r = r_inner + (r_outer - r_inner) * np.sqrt(np.random.rand())
            theta = np.random.rand() * 2 * np.pi
            points.append([r * np.cos(theta), r * np.sin(theta)])
        
        # Outer region
        n_outer_region = int(n_points * 0.4)
        for i in range(n_outer_region):
            r = r_outer + (r_domain - r_outer) * np.sqrt(np.random.rand())
            theta = np.random.rand() * 2 * np.pi
            points.append([r * np.cos(theta), r * np.sin(theta)])
        
        # Add boundary points
        for radius in [r_inner, r_outer, r_domain]:
            n_boundary = 20
            for i in range(n_boundary):
                theta = 2 * np.pi * i / n_boundary
                points.append([radius * np.cos(theta), radius * np.sin(theta)])
        
        points = np.array(points)
    
    # Create triangulation
    tri_obj = Delaunay(points)
    elements = tri_obj.simplices
    
    return points, elements

print("Analytical validation framework initialized!")
print("Available solutions:")
print("  - Infinite wire field and potential")
print("  - Parallel wires configuration")
print("  - Coaxial cable field distribution")

In [ ]:
# ---------- 1.1 Infinite Wire Validation ----------
def solve_infinite_wire_fea(n_points=500, I=100.0, domain_radius=2.0):
    """
    Solve infinite wire problem using FEA for validation.
    
    Parameters:
    -----------
    n_points : int
        Number of mesh points
    I : float
        Current [A]
    domain_radius : float
        Domain radius [m]
    
    Returns:
    --------
    points, elements, A_z, Bx, By, B_magnitude : arrays
        FEA solution
    """
    # Create mesh
    points, elements = create_validation_mesh('circular', n_points, radius=domain_radius)
    
    # Define current source region (small circle at center)
    source_radius = 0.1
    J0 = I / (np.pi * source_radius**2)  # Current density
    
    # Mark source elements
    is_source = []
    for elem in elements:
        centroid = np.mean(points[elem], axis=1)
        r = np.linalg.norm(centroid)
        is_source.append(r < source_radius)
    is_source = np.array(is_source)
    
    # Solve FEA
    n_nodes = len(points)
    K = lil_matrix((n_nodes, n_nodes))
    f = np.zeros(n_nodes)
    mu0 = 4 * np.pi * 1e-7
    
    # Assembly
    for i, elem in enumerate(elements):
        coords = points[elem]
        x = coords[:, 0]
        y = coords[:, 1]
        
        # Element area
        A_e = 0.5 * abs(np.linalg.det(np.array([
            [1, x[0], y[0]],
            [1, x[1], y[1]],
            [1, x[2], y[2]]
        ])))
        
        # Shape function gradients
        b = np.array([y[1] - y[2], y[2] - y[0], y[0] - y[1]])
        c = np.array([x[2] - x[1], x[0] - x[2], x[1] - x[0]])
        B = np.array([b, c]) / (2 * A_e)
        
        # Element stiffness matrix
        Ke = (1 / mu0) * A_e * (B.T @ B)
        
        # Element load vector
        Jz = J0 if is_source[i] else 0.0
        fe = np.ones(3) * Jz * A_e / 3.0
        
        # Assemble
        for j in range(3):
            for k in range(3):
                K[elem[j], elem[k]] += Ke[j, k]
            f[elem[j]] += fe[j]
    
    # Apply boundary conditions (A = 0 at outer boundary)
    boundary_nodes = []
    tolerance = 0.05
    for i, point in enumerate(points):
        if abs(np.linalg.norm(point) - domain_radius) < tolerance:
            boundary_nodes.append(i)
    
    # Penalty method for Dirichlet BC
    penalty = 1e12
    for node in boundary_nodes:
        K[node, node] += penalty
        f[node] += 0  # A = 0 at boundary
    
    # Solve
    K_csr = K.tocsr()
    A_z = spsolve(K_csr, f)
    
    # Compute magnetic field
    Bx = np.zeros(n_nodes)
    By = np.zeros(n_nodes)
    
    for elem in elements:
        coords = points[elem]
        x = coords[:, 0]
        y = coords[:, 1]
        
        A_e = 0.5 * abs(np.linalg.det(np.array([
            [1, x[0], y[0]],
            [1, x[1], y[1]],
            [1, x[2], y[2]]
        ])))
        
        b = np.array([y[1] - y[2], y[2] - y[0], y[0] - y[1]])
        c = np.array([x[2] - x[1], x[0] - x[2], x[1] - x[0]])
        
        dA_dx = np.dot(A_z[elem], b) / (2 * A_e)
        dA_dy = np.dot(A_z[elem], c) / (2 * A_e)
        
        Bx[elem] += dA_dy / 3
        By[elem] -= dA_dx / 3
    
    B_magnitude = np.sqrt(Bx**2 + By**2)
    
    return points, elements, A_z, Bx, By, B_magnitude

def validate_infinite_wire():
    """
    Validate FEA solution against analytical infinite wire solution.
    """
    print("=== Infinite Wire Validation ===")
    
    # Parameters
    I = 100.0  # Current [A]
    mu0 = 4 * np.pi * 1e-7
    
    # Solve with different mesh resolutions
    resolutions = [200, 500, 1000]
    errors = []
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Infinite Wire Validation: FEA vs Analytical Solution', fontsize=16)
    
    for idx, n_points in enumerate(resolutions):
        print(f"\nSolving with {n_points} mesh points...")
        start_time = time.time()
        
        # FEA solution
        points, elements, A_z, Bx, By, B_mag = solve_infinite_wire_fea(n_points, I)
        
        solve_time = time.time() - start_time
        print(f"  Solve time: {solve_time:.3f} s")
        
        # Extract radial profile
        r_values = []
        B_values = []
        
        for point, B in zip(points, B_mag):
            r = np.linalg.norm(point)
            if r > 0.15 and r < 1.8:  # Avoid source and boundary regions
                r_values.append(r)
                B_values.append(B)
        
        # Sort by radius
        sort_idx = np.argsort(r_values)
        r_sorted = np.array(r_values)[sort_idx]
        B_sorted = np.array(B_values)[sort_idx]
        
        # Analytical solution
        B_analytical = AnalyticalSolutions.infinite_wire_field(r_sorted, I, mu0)
        
        # Calculate error
        relative_error = np.abs(B_sorted - B_analytical) / B_analytical * 100
        avg_error = np.mean(relative_error)
        max_error = np.max(relative_error)
        errors.append(avg_error)
        
        print(f"  Average relative error: {avg_error:.2f}%")
        print(f"  Maximum relative error: {max_error:.2f}%")
        
        # Plot comparison
        ax = axes[0, idx]
        ax.semilogx(r_sorted, B_sorted, 'b-', linewidth=2, label='FEA')
        ax.semilogx(r_sorted, B_analytical, 'r--', linewidth=2, label='Analytical')
        ax.set_xlabel('Radius [m]')
        ax.set_ylabel('|B| [T]')
        ax.set_title(f'Radial Field Profile\n({n_points} points)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Plot error
        ax = axes[1, idx]
        ax.semilogx(r_sorted, relative_error, 'g-', linewidth=2)
        ax.fill_between(r_sorted, 0, relative_error, alpha=0.3, color='green')
        ax.set_xlabel('Radius [m]')
        ax.set_ylabel('Relative Error [%]')
        ax.set_title(f'Relative Error\n(Avg: {avg_error:.2f}%)')
        ax.grid(True, alpha=0.3)
        
        # Add statistics text
        stats_text = (f'Elements: {len(elements)}\n'
                      f'Solve time: {solve_time:.3f}s\n'
                      f'Max error: {max_error:.2f}%')
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, 
                verticalalignment='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    # Convergence analysis
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    
    # Error convergence
    ax[0].loglog(resolutions, errors, 'bo-', linewidth=2, markersize=8)
    ax[0].set_xlabel('Number of Mesh Points')
    ax[0].set_ylabel('Average Relative Error [%]')
    ax[0].set_title('Convergence Analysis')
    ax[0].grid(True, alpha=0.3, which='both')
    
    # Add convergence rate
    if len(errors) >= 2:
        rate = np.log(errors[-1]/errors[0]) / np.log(resolutions[-1]/resolutions[0])
        ax[0].text(0.05, 0.95, f'Convergence rate: {rate:.2f}', 
                  transform=ax[0].transAxes, verticalalignment='top',
                  bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    # Field magnitude comparison (using finest mesh)
    points_fine, elements_fine, A_z_fine, Bx_fine, By_fine, B_mag_fine = solve_infinite_wire_fea(1000, I)
    
    # Create 2D field comparison
    xi = np.linspace(-2, 2, 50)
    yi = np.linspace(-2, 2, 50)
    Xi, Yi = np.meshgrid(xi, yi)
    
    # Interpolate FEA results
    B_fea_interp = griddata(points_fine, B_mag_fine, (Xi, Yi), method='linear', fill_value=0)
    
    # Analytical field
    r_grid = np.sqrt(Xi**2 + Yi**2)
    B_analytical_grid = AnalyticalSolutions.infinite_wire_field(r_grid, I, mu0)
    
    # Mask outside domain
    mask = r_grid > 2.0
    B_fea_interp[mask] = np.nan
    B_analytical_grid[mask] = np.nan
    
    # Plot difference
    diff = np.abs(B_fea_interp - B_analytical_grid)
    diff_plot = ax[1].contourf(Xi, Yi, diff, levels=20, cmap='Reds')
    fig.colorbar(diff_plot, ax=ax[1], label='|B_FEA - B_analytical| [T]')
    
    # Add wire location
    wire_circle = Circle((0, 0), 0.1, fill=False, edgecolor='black', linewidth=2)
    ax[1].add_patch(wire_circle)
    
    ax[1].set_xlabel('x [m]')
    ax[1].set_ylabel('y [m]')
    ax[1].set_title('Spatial Error Distribution')
    ax[1].set_aspect('equal')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n=== Validation Summary ===")
    print(f"Expected convergence rate ~2.0 (second-order elements)")
    print(f"Measured convergence rate: {rate:.2f}")
    print(f"Error with finest mesh: {errors[-1]:.2f}%")
    
    return errors

# Run infinite wire validation
print("Starting infinite wire validation...")
convergence_errors = validate_infinite_wire()

## 2. Convergence Studies

Systematic convergence studies are essential for understanding the accuracy and reliability of FEA simulations. We'll implement comprehensive convergence analysis frameworks.

In [ ]:
# ---------- 2. Convergence Study Framework ----------
class ConvergenceStudy:
    """
    Framework for systematic convergence studies.
    """
    
    def __init__(self, problem_type='magnetostatic'):
        self.problem_type = problem_type
        self.results = {}
        self.convergence_data = {}
    
    def run_mesh_convergence(self, mesh_sizes, problem_params, solver_func):
        """
        Run convergence study with different mesh sizes.
        
        Parameters:
        -----------
        mesh_sizes : list
            List of mesh sizes (number of points or elements)
        problem_params : dict
            Problem-specific parameters
        solver_func : callable
            Function to solve the problem
        
        Returns:
        --------
        convergence_data : dict
            Convergence data and metrics
        """
        print(f"Running mesh convergence study for {self.problem_type}...")
        
        convergence_data = {
            'mesh_sizes': mesh_sizes,
            'solutions': [],
            'compute_times': [],
            'errors': [],
            'metrics': []
            'memory_usage': []
        }
        
        for i, mesh_size in enumerate(mesh_sizes):
            print(f"  Mesh {i+1}/{len(mesh_sizes)}: {mesh_size} points")
            
            # Time the solution
            start_time = time.time()
            
            try:
                # Solve problem
                solution = solver_func(mesh_size, **problem_params)
                
                compute_time = time.time() - start_time
                
                # Store solution
                convergence_data['solutions'].append(solution)
                convergence_data['compute_times'].append(compute_time)
                
                # Compute metrics
                metrics = self._compute_metrics(solution)
                convergence_data['metrics'].append(metrics)
                
                # Estimate memory usage (simplified)
                memory_mb = mesh_size * 0.001  # Rough estimate
                convergence_data['memory_usage'].append(memory_mb)
                
                print(f"    Compute time: {compute_time:.3f}s")
                print(f"    Max field: {metrics.get('max_field', 0):.4e} T")
                
            except Exception as e:
                print(f"    Error: {e}")
                convergence_data['solutions'].append(None)
                convergence_data['compute_times'].append(np.nan)
                convergence_data['metrics'].append({})
                convergence_data['memory_usage'].append(np.nan)
        
        # Compute errors (relative to finest mesh)
        finest_solution = convergence_data['solutions'][-1]
        if finest_solution is not None:
            for i, solution in enumerate(convergence_data['solutions'][:-1]):
                if solution is not None:
                    error = self._compute_error(solution, finest_solution)
                    convergence_data['errors'].append(error)
                else:
                    convergence_data['errors'].append(np.nan)
            convergence_data['errors'].append(0.0)  # Finest mesh has zero error relative to itself
        
        self.convergence_data = convergence_data
        return convergence_data
    
    def _compute_metrics(self, solution):
        """
        Compute metrics for a solution.
        """
        if solution is None:
            return {}
        
        # Extract field components
        B_magnitude = solution.get('B_magnitude', np.array([]))
        A_z = solution.get('A_z', np.array([]))
        
        metrics = {}
        
        if len(B_magnitude) > 0:
            metrics['max_field'] = np.max(B_magnitude)
            metrics['avg_field'] = np.mean(B_magnitude)
            metrics['std_field'] = np.std(B_magnitude)
            metrics['energy'] = np.sum(B_magnitude**2)  # Simplified energy metric
        
        if len(A_z) > 0:
            metrics['max_potential'] = np.max(A_z)
            metrics['min_potential'] = np.min(A_z)
        
        return metrics
    
    def _compute_error(self, solution_coarse, solution_fine):
        """
        Compute error between coarse and fine solutions.
        """
        if solution_coarse is None or solution_fine is None:
            return np.nan
        
        # Interpolate fine solution to coarse mesh
        points_coarse = solution_coarse['points']
        points_fine = solution_fine['points']
        B_fine = solution_fine['B_magnitude']
        B_coarse = solution_coarse['B_magnitude']
        
        # Interpolate fine solution to coarse points
        try:
            B_fine_interp = griddata(points_fine, B_fine, points_coarse, method='linear', fill_value=0)
            
            # Compute relative error
            with np.errstate(divide='ignore', invalid='ignore'):
                relative_error = np.abs(B_coarse - B_fine_interp) / (np.abs(B_fine_interp) + 1e-10)
                relative_error = relative_error[~np.isnan(relative_error)]
                
                if len(relative_error) > 0:
                    return np.mean(relative_error) * 100  # Percentage
                else:
                    return np.nan
                    
        except Exception as e:
            print(f"    Error computing error metric: {e}")
            return np.nan
    
    def plot_convergence(self):
        """
        Plot convergence results.
        """
        if not self.convergence_data:
            print("No convergence data available. Run convergence study first.")
            return
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle(f'Convergence Study: {self.problem_type}', fontsize=16)
        
        mesh_sizes = self.convergence_data['mesh_sizes']
        
        # 1. Error convergence
        ax = axes[0, 0]
        valid_errors = [e for e in self.convergence_data['errors'] if not np.isnan(e)]
        valid_sizes = [mesh_sizes[i] for i, e in enumerate(self.convergence_data['errors']) if not np.isnan(e)]
        
        if len(valid_errors) > 1:
            ax.loglog(valid_sizes, valid_errors, 'bo-', linewidth=2, markersize=8)
            
            # Fit convergence rate
            if len(valid_errors) >= 2:
                rate = np.log(valid_errors[-1]/valid_errors[0]) / np.log(valid_sizes[-1]/valid_sizes[0])
                ax.text(0.05, 0.95, f'Convergence rate: {rate:.2f}', 
                        transform=ax.transAxes, verticalalignment='top',
                        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
        
        ax.set_xlabel('Mesh Size')
        ax.set_ylabel('Relative Error [%]')
        ax.set_title('Error Convergence')
        ax.grid(True, alpha=0.3, which='both')
        
        # 2. Compute time
        ax = axes[0, 1]
        valid_times = [t for t in self.convergence_data['compute_times'] if not np.isnan(t)]
        valid_sizes_time = [mesh_sizes[i] for i, t in enumerate(self.convergence_data['compute_times']) if not np.isnan(t)]
        
        if len(valid_times) > 0:
            ax.loglog(valid_sizes_time, valid_times, 'ro-', linewidth=2, markersize=8)
            ax.set_xlabel('Mesh Size')
            ax.set_ylabel('Compute Time [s]')
            ax.set_title('Computational Performance')
            ax.grid(True, alpha=0.3, which='both')
        
        # 3. Memory usage
        ax = axes[0, 2]
        valid_memory = [m for m in self.convergence_data['memory_usage'] if not np.isnan(m)]
        valid_sizes_mem = [mesh_sizes[i] for i, m in enumerate(self.convergence_data['memory_usage']) if not np.isnan(m)]
        
        if len(valid_memory) > 0:
            ax.plot(valid_sizes_mem, valid_memory, 'go-', linewidth=2, markersize=8)
            ax.set_xlabel('Mesh Size')
            ax.set_ylabel('Estimated Memory [MB]')
            ax.set_title('Memory Usage')
            ax.grid(True, alpha=0.3)
        
        # 4. Field magnitude convergence
        ax = axes[1, 0]
        max_fields = []
        avg_fields = []
        valid_sizes_field = []
        
        for i, metrics in enumerate(self.convergence_data['metrics']):
            if metrics and 'max_field' in metrics:
                max_fields.append(metrics['max_field'])
                avg_fields.append(metrics['avg_field'])
                valid_sizes_field.append(mesh_sizes[i])
        
        if len(max_fields) > 0:
            ax.semilogx(valid_sizes_field, max_fields, 'b-', linewidth=2, marker='o', label='Max field')
            ax.semilogx(valid_sizes_field, avg_fields, 'r-', linewidth=2, marker='s', label='Avg field')
            ax.set_xlabel('Mesh Size')
            ax.set_ylabel('Field Magnitude [T]')
            ax.set_title('Field Convergence')
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        # 5. Energy convergence
        ax = axes[1, 1]
        energies = []
        valid_sizes_energy = []
        
        for i, metrics in enumerate(self.convergence_data['metrics']):
            if metrics and 'energy' in metrics:
                energies.append(metrics['energy'])
                valid_sizes_energy.append(mesh_sizes[i])
        
        if len(energies) > 1:
            ax.semilogx(valid_sizes_energy, energies, 'mo-', linewidth=2, markersize=8)
            ax.set_xlabel('Mesh Size')
            ax.set_ylabel('Energy Metric')
            ax.set_title('Energy Convergence')
            ax.grid(True, alpha=0.3)
        
        # 6. Efficiency analysis
        ax = axes[1, 2]
        if len(valid_errors) > 1 and len(valid_times) > 1:
            # Error vs compute time
            ax.semilogx(valid_times, valid_errors, 'co-', linewidth=2, markersize=8)
            ax.set_xlabel('Compute Time [s]')
            ax.set_ylabel('Relative Error [%]')
            ax.set_title('Error vs Computational Cost')
            ax.grid(True, alpha=0.3)
            
            # Add efficiency metric
            if len(valid_errors) > 0 and len(valid_times) > 0:
                efficiency = [1/(e*t) for e, t in zip(valid_errors, valid_times) if e > 0 and t > 0]
                if efficiency:
                    ax2 = ax.twiny()
                    ax2.semilogx(efficiency, valid_errors, 'g--', alpha=0.5)
                    ax2.set_xlabel('Efficiency (1/Error/Time)', color='green')
                    ax2.tick_params(axis='x', labelcolor='green')
        
        plt.tight_layout()
        plt.show()
    
    def generate_report(self):
        """
        Generate a convergence study report.
        """
        if not self.convergence_data:
            print("No convergence data available.")
            return
        
        print("\n" + "="*60)
        print(f"CONVERGENCE STUDY REPORT: {self.problem_type.upper()}")
        print("="*60)
        
        print("\nMesh Sizes:", self.convergence_data['mesh_sizes'])
        
        print("\nCompute Times [s]:")
        for i, time in enumerate(self.convergence_data['compute_times']):
            if not np.isnan(time):
                print(f"  Mesh {i+1}: {time:.3f}s")
        
        print("\nRelative Errors [%]:")
        for i, error in enumerate(self.convergence_data['errors']):
            if not np.isnan(error):
                print(f"  Mesh {i+1}: {error:.3f}%")
        
        # Convergence rate
        valid_errors = [e for e in self.convergence_data['errors'] if not np.isnan(e)]
        valid_sizes = [self.convergence_data['mesh_sizes'][i] for i, e in enumerate(self.convergence_data['errors']) if not np.isnan(e)]
        
        if len(valid_errors) >= 2:
            rate = np.log(valid_errors[-1]/valid_errors[0]) / np.log(valid_sizes[-1]/valid_sizes[0])
            print(f"\nConvergence Rate: {rate:.3f}")
            print(f"Expected (linear elements): ~1.0")
            print(f"Expected (quadratic elements): ~2.0")
        
        # Recommendation
        if len(valid_errors) > 0:
            min_error = min(valid_errors)
            if min_error < 1.0:
                print(f"\n✓ Solution converged to <1% error")
            elif min_error < 5.0:
                print(f"\n⚠ Solution converged to <5% error (acceptable for engineering)")
            else:
                print(f"\n✗ Solution did not converge well (error: {min_error:.1f}%)")
        
        print("="*60)

# Example convergence study solver function
def infinite_wire_convergence_solver(n_points, I=100.0, domain_radius=2.0):
    """
    Solver function for convergence study.
    """
    points, elements, A_z, Bx, By, B_magnitude = solve_infinite_wire_fea(n_points, I, domain_radius)
    
    return {
        'points': points,
        'elements': elements,
        'A_z': A_z,
        'Bx': Bx,
        'By': By,
        'B_magnitude': B_magnitude
    }

# Run convergence study
print("\n=== Convergence Study ===")
convergence_study = ConvergenceStudy('magnetostatic_infinite_wire')
mesh_sizes = [200, 400, 600, 800, 1000]
problem_params = {'I': 100.0, 'domain_radius': 2.0}

convergence_data = convergence_study.run_mesh_convergence(
    mesh_sizes, problem_params, infinite_wire_convergence_solver
)

# Plot results
convergence_study.plot_convergence()

# Generate report
convergence_study.generate_report()

## 3. SfePy Implementation

SfePy is a powerful Python library for solving partial differential equations using the finite element method. Let's explore how to implement electromagnetic problems using SfePy and compare it with our custom implementation.

In [ ]:
# ---------- 3. SfePy Examples (if available) ----------
def sfepy_magnetostatic_example():
    """
    Example of solving a magnetostatic problem using SfePy.
    
    This is a demonstration of how SfePy would be used.
    The actual code would require SfePy to be installed.
    """
    if not sfepy_available:
        print("SfePy not available. Showing example implementation pattern.")
        
        # Show the structure of SfePy implementation
        example_code = '''
# Example SfePy Implementation Pattern
import numpy as nm
from sfepy.base.base import Struct
from sfepy.discrete import FieldVariable, Material, Problem
from sfepy.discrete.fem import FEDomain, Field
from sfepy.terms import term_table
from sfepy.solvers.ls import ScipyDirect
from sfepy.solvers.nls import Newton

# Define mesh (would be loaded from file or created programmatically)
mesh = Mesh.from_file('mesh.msh')
domain = FEDomain('domain', mesh)

# Create field for magnetic vector potential
field = Field.from_args('az', nm.float64, 'scalar', domain, approx_order=1)

# Define variables
az = FieldVariable('az', 'unknown', field)
v = FieldVariable('v', 'test', field, primary_var_name='az')

# Define materials
mu = Material('mu', val={'mu': 4*np.pi*1e-7})  # Permeability
current = Material('current', val={'j': 1e6})     # Current density

# Define equation (magnetostatic: div(1/mu * grad(az)) = -j)
integral = Integral('i', order=2)
t1 = Term.new('dw_laplace(mu.val, v, az)', integral, domain, mu=mu, v=v, az=az)
t2 = Term.new('dw_volume_integrate(current.val, v)', integral, domain, current=current, v=v)

# Create problem
eq = Equation('balance', t1 + t2)
eqs = Equations([eq])
pb = Problem('magnetostatic', equations=eqs)

# Set boundary conditions (A = 0 on outer boundary)
pb.set_bcs(ebcs=EssentialBC('gamma_outer', {'az.0': 0.0}))

# Set solver
ls = ScipyDirect({})
nls = Newton({}, lin_solver=ls)
pb.set_solver(nls)

# Solve
state = pb.solve()

# Extract solution
az_solution = state.get_state_in_region(domain.regions['Omega'])
        '''
        
        print(example_code)
        
        return None
    
    else:
        print("SfePy is available! Actual implementation would go here.")
        print("This would include:")
        print("  - Mesh generation or loading")
        print("  - Field and variable definitions")
        print("  - Material property specification")
        print("  - Equation assembly using SfePy terms")
        print("  - Boundary condition application")
        print("  - Linear/nonlinear solver configuration")
        print("  - Solution extraction and post-processing")
        
        return None

# Demonstrate SfePy approach
print("=== SfePy Implementation ===")
sfepy_result = sfepy_magnetostatic_example()

# Comparison with custom implementation
print("\n=== SfePy vs Custom Implementation ===")

comparison_table = [
    ("Ease of Use", "SfePy: High-level interface\nCustom: Low-level control", "SfePy"),
    ("Flexibility", "SfePy: Built-in terms\nCustom: Complete freedom", "Custom"),
    ("Performance", "SfePy: Optimized assembly\nCustom: Can be optimized", "Tie"),
    ("Learning Curve", "SfePy: Steeper\nCustom: Gradual", "Custom"),
    ("Documentation", "SfePy: Comprehensive\nCustom: Self-documented", "SfePy"),
    ("Debugging", "SfePy: Black box\nCustom: Full visibility", "Custom"),
    ("Advanced Features", "SfePy: Built-in\nCustom: Implement manually", "SfePy"),
    ("Research", "SfePy: Rapid prototyping\nCustom: Novel methods", "Custom")
]

print("\nComparison:")
print("Aspect                 | Description                                    | Preferred")
print("-" * 85)
for aspect, description, preferred in comparison_table:
    print(f"{aspect:<22} | {description:<45} | {preferred}")

print("\nRecommendations:")
print("• Use SfePy for: Standard problems, rapid development, complex PDEs")
print("• Use Custom for: Learning, research, novel methods, full control")
print("• Consider Hybrid: Custom prototype → SfePy production")

## 4. FEniCSx Comparison

FEniCSx is another powerful FEA framework that provides a high-level interface for solving PDEs. Let's explore its capabilities for electromagnetic problems.

In [ ]:
# ---------- 4. FEniCSx Examples (if available) ----------
def fenicsx_magnetostatic_example():
    """
    Example of solving a magnetostatic problem using FEniCSx.
    
    This demonstrates the modern FEniCS approach.
    """
    if not fenics_available:
        print("FEniCSx not available. Showing example implementation pattern.")
        
        example_code = '''
# Example FEniCSx Implementation Pattern
import numpy as np
from dolfinx import mesh, fem, default_scalar_type
from dolfinx.fem.petsc import LinearProblem
from mpi4py import MPI
import ufl

# Create mesh
domain = mesh.create_circle(MPI.COMM_WORLD,
                           1.0,  # radius
                           0.05, # cell size
                           mesh.CellType.triangle)

# Define function space
V = fem.functionspace(domain, ("Lagrange", 1))

# Define trial and test functions
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

# Define material properties
mu0 = 4 * np.pi * 1e-7
mu = fem.Constant(domain, default_scalar_type(mu0))

# Define current source
J0 = 1e6  # Current density
current_source = fem.Constant(domain, default_scalar_type(J0))

# Define variational problem
# ∇·(1/μ ∇A_z) = -J
a = (1/mu) * ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx
L = current_source * v * ufl.dx

# Define boundary condition (A = 0 on boundary)
boundary_facets = mesh.locate_entities_boundary(domain, domain.topology.dim - 1, lambda x: np.full(x.shape[1], True))
boundary_dofs = fem.locate_dofs_topological(V, domain.topology.dim - 1, boundary_facets)
bc = fem.dirichletbc(default_scalar_type(0.0), fem.Function(V), boundary_dofs)

# Solve problem
problem = LinearProblem(a, L, bcs=[bc], petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
A_z = problem.solve()

# Compute magnetic field B = curl(A)
B = ufl.as_vector((A_z.dx(1), -A_z.dx(0)))
B_expr = fem.Expression(B, A_z.function_space)
B_magnitude = ufl.sqrt(B[0]**2 + B[1]**2)
B_mag_expr = fem.Expression(B_magnitude, A_z.function_space)

# Post-processing
import pyvista
pyvista.start_xvfb()

# Create VTK file for visualization
with pyvista.Plotter() as plotter:
    grid = pyvista.UnstructuredGrid(*grid.create_vtk_mesh(domain, domain.topology.dim))
    grid["A_z"] = A_z.x.array
    grid["|B|"] = B_mag_expr.compute().x.array
    
    plotter.add_mesh(grid, show_edges=True)
    plotter.show()
        '''
        
        print(example_code)
        
        return None
    
    else:
        print("FEniCSx is available! Modern implementation would go here.")
        print("FEniCSx advantages:")
        print("  - Modern Python interface")
        print("  - Automatic differentiation")
        print("  - High performance with PETSc")
        print("  - Built-in parallel computing")
        print("  - Integration with modern HPC")
        
        return None

# Library comparison
print("\n=== FEniCSx Implementation ===")
fenicsx_result = fenicsx_magnetostatic_example()

# Comprehensive library comparison
print("\n=== Comprehensive Library Comparison ===")

libraries_data = [
    {
        'Name': 'Custom NumPy/SciPy',
        'Learning Curve': 'Gradual',
        'Performance': 'Good (with optimization)',
        'Flexibility': 'Excellent',
        'Documentation': 'Self-written',
        'Ecosystem': 'Basic',
        'Parallel': 'Manual implementation',
        'Best For': 'Learning, research, custom methods'
    },
    {
        'Name': 'SfePy',
        'Learning Curve': 'Moderate',
        'Performance': 'Very Good',
        'Flexibility': 'Good',
        'Documentation': 'Comprehensive',
        'Ecosystem': 'Moderate',
        'Parallel': 'Built-in',
        'Best For': 'Complex PDEs, research'
    },
    {
        'Name': 'FEniCSx',
        'Learning Curve': 'Steep',
        'Performance': 'Excellent',
        'Flexibility': 'Very Good',
        'Documentation': 'Extensive',
        'Ecosystem': 'Rich',
        'Parallel': 'Excellent',
        'Best For': 'Large-scale problems, HPC'
    },
]

# Create comparison table
print("\nLibrary Comparison Matrix:")
print("="*120)
print(f"{'Library':<20} {'Learning':<12} {'Performance':<12} {'Flexibility':<12} {'Parallel':<12} {'Best Use Case'}")
print("-"*120)

for lib in libraries_data:
    print(f"{lib['Name']:<20} {lib['Learning Curve']:<12} {lib['Performance']:<12} ")
    print(f"{'':20} {lib['Flexibility']:<12} {lib['Parallel']:<12} {lib['Best For']}")
    print()

# Decision tree for library selection
print("\n=== Library Selection Guide ===")
print("\nAnswer these questions to choose the right library:")

questions = [
        ("What's your primary goal?", 
         "Learning/Research → Custom\nStandard problems → SFePy\nLarge-scale HPC → FEniCSx"),
        ("What's your experience level?",
         "Beginner → Custom (gradual)\nIntermediate → SFePy\nAdvanced → FEniCSx"),
        ("Do you need parallel computing?",
         "No → All suitable\nYes → FEniCSx (best) or SFePy"),
        ("Do you need custom PDEs?",
         "Yes → Custom or SFePy\nNo → Any suitable"),
        ("What's your problem size?",
         "Small (<10K DOF) → Custom\nMedium (10K-100K) → SFePy\nLarge (>100K) → FEniCSx")
    ]

for question, guidance in questions:
    print(f"\nQ: {question}")
    print(f"A: {guidance}")

print("\n=== Performance Considerations ===")
performance_tips = [
    "Custom implementation: Optimize for specific problem types",
    "SfePy: Use built-in optimized terms and solvers",
    "FEniCSx: Leverage PETSc and automatic differentiation",
    "All: Use appropriate mesh resolution and element order",
    "All: Choose suitable preconditioners and solvers",
    "All: Consider parallel scaling for large problems"
]

for tip in performance_tips:
    print(f"• {tip}")

## 5. Comprehensive Validation Workflow

Let's create a complete validation workflow that can be used for any electromagnetic FEA problem.

In [ ]:
# ---------- 5. Comprehensive Validation Framework ----------
class ValidationFramework:
    """
    Comprehensive framework for validating FEA solutions.
    """
    
    def __init__(self, problem_name="FEA Validation"):
        self.problem_name = problem_name
        self.validation_results = {}
        self.test_cases = []
    
    def add_test_case(self, name, description, fea_solver, analytical_solver=None):
        """
        Add a test case to the validation framework.
        
        Parameters:
        -----------
        name : str
            Test case name
        description : str
            Test case description
        fea_solver : callable
            Function to solve with FEA
        analytical_solver : callable, optional
            Function to get analytical solution
        """
        test_case = {
            'name': name,
            'description': description,
            'fea_solver': fea_solver,
            'analytical_solver': analytical_solver
        }
        self.test_cases.append(test_case)
    
    def run_validation(self, mesh_sizes, **solver_params):
        """
        Run comprehensive validation for all test cases.
        
        Parameters:
        -----------
        mesh_sizes : list
            List of mesh sizes to test
        **solver_params : dict
            Parameters for solvers
        
        Returns:
        --------
        validation_results : dict
            Comprehensive validation results
        """
        print(f"Running validation for {self.problem_name}")
        print(f"Number of test cases: {len(self.test_cases)}")
        print(f"Mesh sizes: {mesh_sizes}")
        
        validation_results = {
            'problem_name': self.problem_name,
            'mesh_sizes': mesh_sizes,
            'test_cases': [],
            'summary': {}
        }
        
        for test_case in self.test_cases:
            print(f"\n--- Test Case: {test_case['name']} ---")
            print(f"Description: {test_case['description']}")
            
            case_results = {
                'name': test_case['name'],
                'description': test_case['description'],
                'mesh_results': [],
                'convergence_data': None,
                'validation_metrics': {}
            }
            
            # Run for each mesh size
            fea_solutions = []
            analytical_solutions = []
            errors = []
            compute_times = []
            
            for mesh_size in mesh_sizes:
                print(f"  Mesh size: {mesh_size}")
                
                # FEA solution
                start_time = time.time()
                fea_solution = test_case['fea_solver'](mesh_size, **solver_params)
                compute_time = time.time() - start_time
                
                fea_solutions.append(fea_solution)
                compute_times.append(compute_time)
                
                # Analytical solution (if available)
                if test_case['analytical_solver']:
                    analytical_solution = test_case['analytical_solver'](
                        fea_solution['points'], **solver_params
                    )
                    analytical_solutions.append(analytical_solution)
                    
                    # Compute error
                    error = self._compute_solution_error(fea_solution, analytical_solution)
                    errors.append(error)
                    
                    print(f"    Error: {error:.3f}%")
                else:
                    analytical_solutions.append(None)
                    errors.append(None)
                
                print(f"    Compute time: {compute_time:.3f}s")
                
                case_results['mesh_results'].append({
                    'mesh_size': mesh_size,
                    'fea_solution': fea_solution,
                    'analytical_solution': analytical_solution,
                    'error': errors[-1],
                    'compute_time': compute_time
                })
            
            # Compute convergence metrics
            if any(errors[i] is not None for i in range(len(errors))):
                valid_errors = [e for e in errors if e is not None]
                valid_sizes = [mesh_sizes[i] for i, e in enumerate(errors) if e is not None]
                
                if len(valid_errors) >= 2:
                    convergence_rate = np.log(valid_errors[-1]/valid_errors[0]) / np.log(valid_sizes[-1]/valid_sizes[0])
                    case_results['convergence_rate'] = convergence_rate
                    
                    # Predict error for infinite mesh
                    log_error_inf = np.log(valid_errors[0]) - convergence_rate * np.log(valid_sizes[0])
                    predicted_error_inf = np.exp(log_error_inf)
                    case_results['predicted_error_infinite'] = predicted_error_inf
            
            # Validation metrics
            case_results['validation_metrics'] = self._compute_validation_metrics(
                fea_solutions, analytical_solutions, errors, compute_times
            )
            
            validation_results['test_cases'].append(case_results)
        
        # Overall summary
        validation_results['summary'] = self._create_summary(validation_results['test_cases'])
        
        self.validation_results = validation_results
        return validation_results
    
    def _compute_solution_error(self, fea_solution, analytical_solution):
        """
        Compute error between FEA and analytical solutions.
        """
        # This is a simplified error computation
        # In practice, you would use more sophisticated error measures
        
        try:
            B_fea = fea_solution['B_magnitude']
            B_analytical = analytical_solution['B_magnitude']
            
            # L2 norm error
            error_l2 = np.linalg.norm(B_fea - B_analytical) / np.linalg.norm(B_analytical) * 100
            
            return error_l2
            
        except Exception as e:
            print(f"Error computing solution error: {e}")
            return np.nan
    
    def _compute_validation_metrics(self, fea_solutions, analytical_solutions, errors, compute_times):
        """
        Compute comprehensive validation metrics.
        """
        metrics = {}
        
        # Error statistics
        valid_errors = [e for e in errors if e is not None]
        if valid_errors:
            metrics['min_error'] = np.min(valid_errors)
            metrics['max_error'] = np.max(valid_errors)
            metrics['avg_error'] = np.mean(valid_errors)
            metrics['std_error'] = np.std(valid_errors)
        
        # Performance metrics
        metrics['min_compute_time'] = np.min(compute_times)
        metrics['max_compute_time'] = np.max(compute_times)
        metrics['avg_compute_time'] = np.mean(compute_times)
        
        # Efficiency metrics
        if valid_errors and compute_times:
            efficiency_scores = [1/(e*t) for e, t in zip(valid_errors, compute_times) if e > 0 and t > 0]
            if efficiency_scores:
                metrics['avg_efficiency'] = np.mean(efficiency_scores)
                metrics['max_efficiency'] = np.max(efficiency_scores)
        
        return metrics
    
    def _create_summary(self, test_cases):
        """
        Create overall validation summary.
        """
        summary = {
            'total_test_cases': len(test_cases),
            'best_convergence': None,
            'fastest_solver': None,
            'most_accurate': None,
            'recommendations': []
        }
        
        # Find best performers
        best_convergence_rate = -np.inf
        min_compute_time = np.inf
        min_error = np.inf
        
        for case in test_cases:
            if 'convergence_rate' in case and case['convergence_rate'] > best_convergence_rate:
                best_convergence_rate = case['convergence_rate']
                summary['best_convergence'] = case['name']
            
            metrics = case.get('validation_metrics', {})
            if 'min_compute_time' in metrics and metrics['min_compute_time'] < min_compute_time:
                min_compute_time = metrics['min_compute_time']
                summary['fastest_solver'] = case['name']
            
            if 'min_error' in metrics and metrics['min_error'] < min_error:
                min_error = metrics['min_error']
                summary['most_accurate'] = case['name']
        
        # Generate recommendations
        recommendations = []
        
        if best_convergence_rate > 1.5:
            recommendations.append("Excellent convergence achieved")
        elif best_convergence_rate > 1.0:
            recommendations.append("Good convergence achieved")
        else:
            recommendations.append("Convergence could be improved")
        
        if min_error < 1.0:
            recommendations.append("High accuracy achieved")
        elif min_error < 5.0:
            recommendations.append("Reasonable accuracy for engineering")
        else:
            recommendations.append("Accuracy needs improvement")
        
        summary['recommendations'] = recommendations
        
        return summary
    
    def generate_report(self):
        """
        Generate comprehensive validation report.
        """
        if not self.validation_results:
            print("No validation results available. Run validation first.")
            return
        
        print("\n" + "="*80)
        print(f"VALIDATION REPORT: {self.validation_results['problem_name']}")
        print("="*80)
        
        print(f"\nTest Cases: {len(self.validation_results['test_cases'])}")
        print(f"Mesh Sizes Tested: {self.validation_results['mesh_sizes']}")
        
        print("\n" + "-"*40)
        print("DETAILED RESULTS")
        print("-"*40)
        
        for case in self.validation_results['test_cases']:
            print(f"\n{case['name']}:")
            print(f"  Description: {case['description']}")
            
            metrics = case.get('validation_metrics', {})
            if 'min_error' in metrics:
                print(f"  Best Accuracy: {metrics['min_error']:.3f}%")
                print(f"  Average Accuracy: {metrics.get('avg_error', 0):.3f}%")
                print(f"  Fastest Solve: {metrics['min_compute_time']:.3f}s")
            
            if 'convergence_rate' in case:
                print(f"  Convergence Rate: {case['convergence_rate']:.3f}")
            
            if 'predicted_error_infinite' in case:
                print(f"  Predicted Error (infinite mesh): {case['predicted_error_infinite']:.3e}%")
        
        print("\n" + "-"*40)
        print("SUMMARY")
        print("-"*40)
        
        summary = self.validation_results['summary']
        print(f"\nBest Convergence: {summary.get('best_convergence', 'N/A')}")
        print(f"Fastest Solver: {summary.get('fastest_solver', 'N/A')}")
        print(f"Most Accurate: {summary.get('most_accurate', 'N/A')}")
        
        print("\nRecommendations:")
        for rec in summary.get('recommendations', []):
            print(f"• {rec}")
        
        print("\n" + "="*80)
        print("END OF VALIDATION REPORT")
        print("="*80)
    
    def plot_validation_results(self):
        """
        Plot comprehensive validation results.
        """
        if not self.validation_results:
            print("No validation results available.")
            return
        
        n_cases = len(self.validation_results['test_cases'])
        if n_cases == 0:
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(f'Validation Results: {self.validation_results["problem_name"]}', fontsize=16)
        
        mesh_sizes = self.validation_results['mesh_sizes']
        
        # 1. Error convergence for all test cases
        ax = axes[0, 0]
        for case in self.validation_results['test_cases']:
            errors = []
            sizes = []
            
            for mesh_result in case['mesh_results']:
                if mesh_result['error'] is not None:
                    errors.append(mesh_result['error'])
                    sizes.append(mesh_result['mesh_size'])
            
            if len(errors) > 1:
                ax.loglog(sizes, errors, 'o-', linewidth=2, markersize=6, label=case['name'])
        
        ax.set_xlabel('Mesh Size')
        ax.set_ylabel('Relative Error [%]')
        ax.set_title('Convergence Comparison')
        ax.legend()
        ax.grid(True, alpha=0.3, which='both')
        
        # 2. Performance comparison
        ax = axes[0, 1]
        for case in self.validation_results['test_cases']:
            times = []
            sizes = []
            
            for mesh_result in case['mesh_results']:
                times.append(mesh_result['compute_time'])
                sizes.append(mesh_result['mesh_size'])
            
            ax.loglog(sizes, times, 's-', linewidth=2, markersize=6, label=case['name'])
        
        ax.set_xlabel('Mesh Size')
        ax.set_ylabel('Compute Time [s]')
        ax.set_title('Performance Comparison')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 3. Accuracy vs. Efficiency
        ax = axes[1, 0]
        for case in self.validation_results['test_cases']:
            errors = []
            times = []
            
            for mesh_result in case['mesh_results']:
                if mesh_result['error'] is not None:
                    errors.append(mesh_result['error'])
                    times.append(mesh_result['compute_time'])
            
            if len(errors) > 0:
                ax.semilogx(times, errors, 'o-', linewidth=2, markersize=6, label=case['name'])
        
        ax.set_xlabel('Compute Time [s]')
        ax.set_ylabel('Relative Error [%]')
        ax.set_title('Accuracy vs. Efficiency')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 4. Summary metrics
        ax = axes[1, 1]
        
        case_names = [case['name'] for case in self.validation_results['test_cases']]
        min_errors = []
        min_times = []
        
        for case in self.validation_results['test_cases']:
            metrics = case.get('validation_metrics', {})
            min_errors.append(metrics.get('min_error', np.nan))
            min_times.append(metrics.get('min_compute_time', np.nan))
        
        x_pos = np.arange(len(case_names))
        width = 0.35
        
        # Create second y-axis for times
        ax2 = ax.twinx()
        
        bars1 = ax.bar(x_pos - width/2, min_errors, width, label='Min Error', alpha=0.7)
        bars2 = ax2.bar(x_pos + width/2, min_times, width, label='Min Time', alpha=0.7, color='orange')
        
        ax.set_xlabel('Test Case')
        ax.set_ylabel('Min Error [%]', color='blue')
        ax2.set_ylabel('Min Time [s]', color='orange')
        ax.set_title('Best Performance Metrics')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(case_names, rotation=45, ha='right')
        ax.tick_params(axis='y', labelcolor='blue')
        ax2.tick_params(axis='y', labelcolor='orange')
        
        # Combined legend
        lines = [bars1, bars2]
        labels = [l.get_label() for l in lines]
        ax.legend(lines, labels, loc='upper center')
        
        plt.tight_layout()
        plt.show()

# Demonstrate validation framework
print("\n=== Validation Framework Demo ===")

# Create validation framework
validator = ValidationFramework("Magnetostatic Validation")

# Add test cases
validator.add_test_case(
    "Infinite Wire",
    "Magnetic field of infinite current-carrying wire",
    infinite_wire_convergence_solver,
    lambda points, **params: {
        'B_magnitude': AnalyticalSolutions.infinite_wire_field(
            np.linalg.norm(points, axis=1),
            params.get('I', 100.0)
        )
    }
)

# Run validation
mesh_sizes = [200, 400, 600, 800]
validation_results = validator.run_validation(mesh_sizes, I=100.0)

# Generate report
validator.generate_report()

# Plot results
validator.plot_validation_results()

## 6. Best Practices for Validation

Let's summarize the key principles and best practices for validation of electromagnetic FEA simulations.

In [ ]:
# ---------- 6. Best Practices Summary ----------
print("=" * 80)
print("VALIDATION & LIBRARY COMPARISONS: BEST PRACTICES")
print("=" * 80)

print("\n1. VALIDATION HIERARCHY:")
validation_hierarchy = [
    "Level 1: Code verification (debugging, unit tests)",
    "Level 2: Method verification (analytical solutions)",
    "Level 3: Benchmark problems (standard test cases)",
    "Level 4: Experimental validation (real-world data)",
    "Level 5: Peer validation (independent implementation)"
]

for i, level in enumerate(validation_hierarchy, 1):
    print(f"   {i}. {level}")

print("\n2. ANALYTICAL SOLUTION REQUIREMENTS:")
analytical_requirements = [
    ("Mathematical Exactness", "No approximations or simplifications"),
    ("Boundary Conditions", "Must match FEA implementation exactly"),
    ("Material Properties", "Must use identical material models"),
    ("Geometry", "Must represent identical problem geometry"),
    ("Domain Definition", "Must include same solution domain")
    ("Units Consistency", "Must use consistent unit systems")
    ("Numerical Precision", "Must maintain sufficient precision")
    ("Singularity Handling", "Must properly handle singularities")
]

print("   Requirement               | Importance")
print("   " + "-" * 58)
for requirement, importance in analytical_requirements:
    print(f"   {requirement:<26} | {importance}")

print("\n3. CONVERGENCE STUDY GUIDELINES:")
convergence_guidelines = [
    "Use at least 3 different mesh resolutions",
    "Ensure systematic refinement (not random)",
    "Monitor multiple quantities (field, energy, forces)",
    "Check both absolute and relative errors",
    "Verify convergence order matches theoretical expectations",
    "Identify and address any divergence or oscillation",
    "Document convergence criteria and tolerances"
]

for i, guideline in enumerate(convergence_guidelines, 1):
    print(f"   {i}. {guideline}")

print("\n4. ERROR METRICS AND THEIR USES:")
error_metrics = [
    ("L2 Norm Error", "Overall field accuracy"),
    ("L∞ Norm Error", "Maximum local error"),
    ("Relative Error", "Percentage-based comparison"),
    ("Energy Norm Error", "Energy-based accuracy"),
    ("Point-wise Error", "Local accuracy assessment"),
    ("Integrated Error", "Global quantity accuracy"),
    ("Flux Conservation Error", "Physical law verification")
]

print("   Metric                | Application")
print("   " + "-" * 45)
for metric, application in error_metrics:
    print(f"   {metric:<22} | {application}")

print("\n5. LIBRARY SELECTION CRITERIA:")
selection_criteria = [
    ("Problem Complexity", "Simple → Custom, Complex → SFePy/FEniCSx"),
    ("Performance Requirements", "Speed → Custom, Scale → FEniCSx"),
    ("Development Time", "Rapid → SfePy, Learning → Custom"),
    ("Team Expertise", "Beginner → Custom, Expert → Any"),
    ("Support Requirements", "Community → SfePy/FEniCSx"),
    ("Customization Needs", "High → Custom, Low → Libraries"),
    ("Long-term Maintenance", "Standard → Libraries")
]

print("   Criterion               | Recommendation")
print("   " + "-" * 58)
for criterion, recommendation in selection_criteria:
    print(f"   {criterion:<23} | {recommendation}")

print("\n6. DOCUMENTATION AND REPRODUCIBILITY:")
documentation_practices = [
    "• Record all mesh parameters and generation methods",
    "• Document boundary conditions and their implementation",
    "• Archive input files and parameter sets",
    "• Use version control for all analysis scripts",
    "• Include convergence studies in documentation",
    "• Provide clear validation methodologies",
    "• Archive both input and output data",
    "• Document any approximations or assumptions"
]

for practice in documentation_practices:
    print(f"{practice}")

print("\n7. QUALITY ASSURANCE CHECKLIST:")
qa_checklist = [
    "✓ Mesh independence verified",
    "✓ Boundary conditions correctly implemented",
    "✓ Material properties properly defined",
    "✓ Units consistent throughout simulation",
    "✓ Convergence achieved and documented",
    "✓ Results validated against analytical solutions",
    "✓ Physical laws (e.g., ∇·B = 0) satisfied",
    "✓ Error metrics within acceptable limits",
    "✓ Solution symmetric when expected",
    "✓ Energy conservation verified",
    "✓ Results reproducible by independent methods"
]

for item in qa_checklist:
    print(f"{item}")

print("\n" + "=" * 80)
print("CRITICAL SUCCESS FACTORS:")
print("• Systematic validation approach")
print("• Multiple verification methods")
print("• Clear documentation and reproducibility")
print("• Understanding of tool limitations")
print("• Regular quality assurance processes")
print("=" * 80)

print("\n📊 FINAL SUMMARY OF APPENDIX A:")
print("   ✅ Complete FEA implementation from scratch")
print("   ✅ Advanced mesh generation techniques")
print("   ✅ Multi-material boundary conditions")
print("   ✅ Comprehensive post-processing and visualization")
print("   ✅ Robust validation and comparison framework")
print("   ✅ Library ecosystem overview and guidance")

print("\n🎯 READY FOR: Production-level electromagnetic FEA analysis!")
print("\nNext Steps:")
print("• Apply to your specific electromagnetic problems")
print("• Extend to time-dependent and nonlinear problems")
print("• Integrate with optimization workflows")
print("• Contribute to the FEA community")

## Summary

This notebook provided comprehensive coverage of validation techniques and library comparisons for electromagnetic FEA. We explored:

### ✅ Analytical Validation Framework

1. **Analytical Solutions**: Infinite wire, parallel wires, coaxial cable configurations
2. **Systematic Validation**: Complete framework for comparing FEA with analytical results
3. **Error Analysis**: Multiple error metrics and convergence assessment methods
4. **Benchmark Problems**: Standard test cases for method verification

### 🔬 Convergence Study Methodology

- **Mesh Independence Studies**: Systematic refinement and error assessment
- **Convergence Rate Analysis**: Theoretical vs. observed convergence behavior
- **Performance Metrics**: Computational efficiency vs. accuracy trade-offs
- **Quality Assurance**: Comprehensive validation workflows

### 📚 Library Ecosystem Overview

- **Custom Implementation**: Full control and understanding
- **SfePy**: Specialized PDE solving with comprehensive term library
- **FEniCSx**: High-performance modern FEA framework
- **Selection Criteria**: Guidelines for choosing appropriate tools

### 🎯 Best Practices and Standards

- **Validation Hierarchy**: Multi-level verification approach
- **Documentation Standards**: Reproducible and well-documented workflows
- **Quality Assurance**: Systematic quality check procedures
- **Error Management**: Comprehensive error analysis and reporting

### 💡 Critical Insights

- **Validation is Essential**: No FEA implementation is complete without validation
- **Multiple Validation Methods**: Cross-validation using different approaches increases confidence
- **Library Choice Matters**: Different tools excel at different problem types
- **Systematic Approach**: Structured validation frameworks ensure reliability

The validation techniques and library comparisons presented here provide a solid foundation for ensuring the accuracy and reliability of electromagnetic FEA simulations. These methods are essential for both research investigations and engineering applications where solution accuracy is critical.

---

**🎉 Congratulations! You have completed Appendix A: Finite Element Analysis Supplementary Material**

You now have:
- A complete FEA implementation from scratch using NumPy/SciPy
- Advanced mesh generation and boundary condition techniques
- Comprehensive post-processing and visualization methods
- Robust validation and comparison frameworks
- Deep understanding of the Python FEA ecosystem

You are ready to tackle complex electromagnetic problems with confidence! 🚀